In [47]:
import pandas as pd
import re

In [48]:
PATH_TO_DATASETS = 'Raw_Datasets/MMG'

CANONICAL_COLUMNS = {
    'FIPS':                         'FIPS',
    'State':                        'State',
    'County, State':                'County, State',
    'FI Rate':                      'FI Rate',
    'Num FI Persons':               'Num FI Persons',
    'Low Threshold in state':       'Low Threshold in state',
    'Low Threshold Type':           'Low Threshold Type',
    'High Threshold in state':      'High Threshold in state',
    'High Threshold Type':          'High Threshold Type',
    '% FI ≤ Low Threshold':        '% FI ≤ Low Threshold',
    '% FI Btwn Thresholds':        '% FI Btwn Thresholds',
    '% FI > High Threshold':       '% FI > High Threshold',
    'Child FI Rate':                'Child FI Rate',
    'Num FI Children':              'Num FI Children',
    '% FI Children Below 185 FPL': '% FI Children Below 185 FPL',
    '% FI Children Above 185 FPL': '% FI Children Above 185 FPL',
    'Cost Per Meal':                'Cost Per Meal',
    'Weighted Annual Food Budget Shortfall': 'Weighted Annual Food Budget Shortfall',
}

def normalize_columns(df, year):
    """Rename messy year-specific columns to canonical names."""
    yr = str(year)
    rename_map = {}

    for col in df.columns:
        c = col.strip()
        if c == 'State Name':
            rename_map[col] = 'County, State'
        elif re.search(rf'{yr} food insecurity rate', c, re.IGNORECASE):
            rename_map[col] = 'FI Rate'
        elif re.search(r'(number of|#\s*of) food insecure persons', c, re.IGNORECASE):
            rename_map[col] = 'Num FI Persons'
        elif re.search(rf'{yr} child food insecurity rate', c, re.IGNORECASE):
            rename_map[col] = 'Child FI Rate'
        elif re.search(r'(number of|#\s*of) food insecure children', c, re.IGNORECASE):
            rename_map[col] = 'Num FI Children'
        elif re.search(r'food insecure children.+below 185', c, re.IGNORECASE):
            rename_map[col] = '% FI Children Below 185 FPL'
        elif re.search(r'food insecure children.+above 185', c, re.IGNORECASE):
            rename_map[col] = '% FI Children Above 185 FPL'
        elif re.search(rf'{yr} cost per meal', c, re.IGNORECASE):
            rename_map[col] = 'Cost Per Meal'
        elif re.search(r'weighted annual food budget shortfall', c, re.IGNORECASE):
            rename_map[col] = 'Weighted Annual Food Budget Shortfall'
        elif c != col:
            rename_map[col] = c
    return df.rename(columns=rename_map)


sets = []
for year in range(10, 18):
    full_year = 2000 + year
    df = pd.read_excel(io=f'{PATH_TO_DATASETS}/MMG20{year+2}_20{year}Data_ToShare.xlsx')
    if year in range(11,14):
        df = pd.read_excel(io=f'{PATH_TO_DATASETS}/MMG20{year+2}_20{year}Data_ToShare.xlsx', sheet_name=f'{full_year} County')
    df = normalize_columns(df, full_year)
    df['Year'] = year
    df['Name'] = df['County, State'].apply(lambda x: x.split(',')[0])
    sets.append(df)

combined = pd.concat(sets, ignore_index=True)
print(combined.columns.tolist())
print(combined.shape)

['FIPS', 'State', 'County, State', 'FI Rate', 'Num FI Persons', 'Low Threshold in state', 'Low Threshold Type', 'High Threshold in state', 'High Threshold Type', '% FI ≤ Low Threshold', '% FI Btwn Thresholds', '% FI > High Threshold', 'Child FI Rate', 'Num FI Children', '% FI Children Below 185 FPL', '% FI Children Above 185 FPL', 'Cost Per Meal', 'Weighted Annual Food Budget Shortfall', 'Year', 'Name']
(25140, 20)


In [49]:
combined

,FIPS,State,"County, State",FI Rate,Num FI Persons,Low Threshold in state,Low Threshold Type,High Threshold in state,High Threshold Type,% FI ≤ Low Threshold,% FI Btwn Thresholds,% FI > High Threshold,Child FI Rate,Num FI Children,% FI Children Below 185 FPL,% FI Children Above 185 FPL,Cost Per Meal,Weighted Annual Food Budget Shortfall,Year,Name
0,1001,AL,"Autauga County, Alabama",0.134,7140,1.3,SNAP,1.85,Other Nutrition Program,0.327,0.208,0.465,0.203,2980.0,0.51,0.49,2.58,3170830,10,Autauga County
1,1003,AL,"Baldwin County, Alabama",0.134,23570,1.3,SNAP,1.85,Other Nutrition Program,0.347,0.287,0.366,0.238,9720.0,0.59,0.41,2.64,10710730,10,Baldwin County
2,1005,AL,"Barbour County, Alabama",0.232,6440,1.3,SNAP,1.85,Other Nutrition Program,0.479,0.171,0.350,0.258,1600.0,0.87,0.13,2.53,2804540,10,Barbour County
3,1007,AL,"Bibb County, Alabama",0.157,3550,1.3,SNAP,1.85,Other Nutrition Program,0.358,0.288,0.354,0.249,1300.0,0.64,0.36,2.55,1558200,10,Bibb County
4,1009,AL,"Blount County, Alabama",0.126,7160,1.3,SNAP,1.85,Other Nutrition Program,0.410,0.305,0.285,0.254,3540.0,0.53,0.47,2.50,3081120,10,Blount County
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25135,56037,WY,"Sweetwater County, Wyoming",0.107,4750,1.3,SNAP,1.85,Other Nutrition Program,0.430,0.128,0.442,0.17,2030.0,0.51,0.49,3.14,2542000,17,Sweetwater County
25136,56039,WY,"Teton County, Wyoming",0.097,2220,1.3,SNAP,1.85,Other Nutrition Program,0.360,0.167,0.473,0.117,520.0,0.56,0.44,4.20,1592000,17,Teton County
25137,56041,WY,"Uinta County, Wyoming",0.128,2660,1.3,SNAP,1.85,Other Nutrition Program,0.562,0.074,0.365,0.189,1160.0,0.64,0.36,2.95,1340000,17,Uinta County
25138,56043,WY,"Washakie County, Wyoming",0.112,920,1.3,SNAP,1.85,Other Nutrition Program,0.505,0.172,0.323,0.174,350.0,0.74,0.27,3.16,497000,17,Washakie County


In [50]:
combined.to_csv("Cleaned_Datasets/MMG.csv", index=False)